# Thesis presentation figures
Generates every figure for the thesis talk. Run cells top-to-bottom; each section is independent and skips gracefully if its input is missing.

In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

Working directory: /groups/nils/members/andras/scrna_project


In [2]:
import sys, glob, warnings
import sqlite3
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display
warnings.filterwarnings('ignore', category=DeprecationWarning)

# ── Paths ──────────────────────────────────────────────────────────────────
DB_PATH   = 'results/variants.db'
DEMUX_DIR = 'results/demux'
FIG_DIR   = 'figures/presentation'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(f'{FIG_DIR}/mmd', exist_ok=True)

# ── Read-only DB ────────────────────────────────────────────────────────────
def query(sql, params=()):
    with sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True) as conn:
        return pd.read_sql_query(sql, conn, params=params)

# ── Palette ─────────────────────────────────────────────────────────────────
C = dict(
    dark_teal   = '#1B3A3F',
    accent_teal = '#1F7A7A',
    wes         = '#6D597A',
    same_donor  = '#E76F51',
    diff_donor  = '#457B9D',
    grey        = '#B0B7BB',
    red         = '#E63946',
    green       = '#2D6A4F',
    orange      = '#F4A261',
)
W, H, FS = 1600, 900, 18

# ── Layout helper ───────────────────────────────────────────────────────────
def _style(fig, title=None, **kw):
    layout = dict(
        paper_bgcolor='white', plot_bgcolor='white',
        font=dict(size=FS, family='Arial, sans-serif'),
        margin=dict(l=90, r=60, t=80 if title else 40, b=90),
        showlegend=kw.pop('showlegend', True),
    )
    if title:
        layout['title'] = dict(text=title, font=dict(size=FS+2))
    layout.update(kw)
    fig.update_layout(**layout)
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#EEEEEE', zeroline=False)
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#EEEEEE', zeroline=False)
    return fig

def save(fig, name, w=W, h=H, show=True):
    base = f'{FIG_DIR}/{name}'
    fig.update_layout(width=w, height=h)
    try:
        fig.write_image(f'{base}.png', scale=2)
        fig.write_image(f'{base}.svg')
        print(f'  ✓ {base}.png / .svg')
    except Exception as e:
        print(f'  Image export failed ({e}) — saving HTML')
        fig.write_html(f'{base}.html')
    if show:
        fig.show()
    return base

# ── Run table ───────────────────────────────────────────────────────────────
all_runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt
    FROM runs r JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")
all_runs['label'] = all_runs['cell_line'].fillna(all_runs['sample'])
all_runs['dtype'] = all_runs['run_id'].apply(
    lambda x: 'WES' if '_wes_' in x.lower() else 'sc/bulk'
)
sc_runs  = all_runs[all_runs['dtype'] == 'sc/bulk'].copy().reset_index(drop=True)
wes_runs = all_runs[all_runs['dtype'] == 'WES'].copy().reset_index(drop=True)
print(f'DB: {len(sc_runs)} sc/bulk runs, {len(wes_runs)} WES runs')

DB: 44 sc/bulk runs, 10 WES runs


## Figure 1 — Pipeline flowcharts (slides 3, 4, 7)
**Slide message:** three panels showing the overall data flow, the reference-profiling path, and the demultiplexing path.

Writes `.mmd` files. Renders with `mmdc` if available.

In [3]:
MMDS = {
    'fig1a_overview': """\
flowchart LR
    A(["FASTQ / BAM<br>input data"])
    B["Genotype at<br>common SNPs<br>(1000G, AF > 5%)"]
    C[("SQLite DB<br>variants.db")]
    D["Cell-line<br>assignment<br>final_assignments.tsv"]
    A --> B --> C --> D
    style C fill:#1F7A7A,color:#fff
    style D fill:#1B3A3F,color:#fff
""",
    'fig1b_profiling': """\
flowchart TD
    A(["FASTQ<br>sc RNA-seq or WES"])
    B1["STAR align<br>(sc / bulk)"]
    B2["BWA-MEM align<br>(WES / WGS)"]
    C["samtools sort + index"]
    QC{"mapping rate<br>≥ 85%?"}
    D["bcftools mpileup<br>SNP panel positions only"]
    E["bcftools call<br>0/0 · 0/1 · 1/1"]
    F["bcftools filter<br>DP ≥ 10 · QUAL ≥ 30"]
    G[("SQLite DB")]
    A --> B1 & B2 --> C --> QC
    QC -->|pass| D --> E --> F --> G
    QC -->|fail| STOP(["Pipeline abort"])
    style QC fill:#E63946,color:#fff
    style STOP fill:#E63946,color:#fff
    style G fill:#1F7A7A,color:#fff
""",
    'fig1c_demux': """\
flowchart TD
    CR(["CellRanger BAM<br>+ barcodes.tsv.gz"])
    CS["cellsnp-lite<br>per-cell pileup"]
    VG["Vireo<br>donor clustering"]
    VU["genotype-free<br>(unguided)"]
    GD["reference-guided<br>(known cell lines)"]
    MV["match_vireo.py<br>concordance + Hungarian"]
    MG["donor names<br>= cell line names"]
    ME["merge_demux.py"]
    FA(["final_assignments.tsv<br>barcode · cell_line · confidence"])
    CR --> CS --> VG
    VG --> VU --> MV --> ME
    VG --> GD --> MG --> ME
    ME --> FA
    style FA fill:#1B3A3F,color:#fff
    style VU fill:#457B9D,color:#fff
    style GD fill:#1F7A7A,color:#fff
""",
}

for name, content in MMDS.items():
    path = f'{FIG_DIR}/mmd/{name}.mmd'
    with open(path, 'w') as f:
        f.write(content)
    print(f'Wrote {path}')

# Try to render with mmdc
import shutil
mmdc = shutil.which('mmdc')
if not mmdc:
    for candidate in ['/usr/local/bin/mmdc', '/usr/bin/mmdc']:
        if os.path.exists(candidate):
            mmdc = candidate
            break

if mmdc:
    for name in MMDS:
        inp = f'{FIG_DIR}/mmd/{name}.mmd'
        out = f'{FIG_DIR}/{name}.svg'
        ret = os.system(f'{mmdc} -i {inp} -o {out} -b white 2>/dev/null')
        print(f'  {name}: {"OK" if ret == 0 else "FAILED"}')
else:
    print('mmdc not found — .mmd files written to figures/presentation/mmd/')
    print('Render with: npm install -g @mermaid-js/mermaid-cli && mmdc -i <file>.mmd -o <file>.svg -b white')

Wrote figures/presentation/mmd/fig1a_overview.mmd
Wrote figures/presentation/mmd/fig1b_profiling.mmd
Wrote figures/presentation/mmd/fig1c_demux.mmd
mmdc not found — .mmd files written to figures/presentation/mmd/
Render with: npm install -g @mermaid-js/mermaid-cli && mmdc -i <file>.mmd -o <file>.svg -b white


## Figure 2 — 0/0 genotype schematic (slide 5)
**Slide message:** Storing 0/0 (ref/ref) calls is essential — without them, a wrong reference with few ALTs can appear to match perfectly because only shared ALT positions enter the denominator.

In [4]:
# 12 SNP positions × 3 rows.  Values: 0=missing, 1=REF(0/0), 2=ALT present
# Design: Query has ALT at pos 1-4 and 9-11, REF at 5-8, missing at 12
# Ref A (true match, backfilled):  same profile as Query at 1-11, REF at 12
# Ref B (wrong line, legacy):       ALT only at 1 and 9, missing everywhere else

# WITHOUT 0/0 (only ALT positions in Ref B): Ref B ALTs={1,9} -> Q=ALT at both -> 2/2=100%
#   Ref A ALTs={1-4,9-11} -> Q=ALT at all 7 -> 7/7=100%  => INDISTINGUISHABLE
# WITH 0/0 (Ref A backfilled, Ref B also backfilled):
#   Ref A: shared=1-11 (12 missing in both) -> 11/11=100%
#   Ref B backfilled: pos1=A/A✓, pos2-4=A/0 3 mismatches, pos5-8=0/0 4 matches,
#                     pos9=A/A✓, pos10-11=A/0 2 mismatches -> 6/11=55%

MISS, REF, ALT = 0, 1, 2

positions = list(range(1, 13))
rows = {
    'Query\n(Vireo donor)':         [ALT, ALT, ALT, ALT, REF, REF, REF, REF, ALT, ALT, ALT, MISS],
    'Ref A — true match\n(with 0/0 backfilled)':  [ALT, ALT, ALT, ALT, REF, REF, REF, REF, ALT, ALT, ALT, REF],
    'Ref B — wrong line\n(legacy, only ALT calls)': [ALT, MISS, MISS, MISS, MISS, MISS, MISS, MISS, ALT, MISS, MISS, MISS],
}
row_labels = list(rows.keys())
mat = np.array([rows[r] for r in row_labels], dtype=float)
mat_display = mat.copy()

# Color text for annotations
val_text = {
    MISS: 'missing',
    REF:  '0/0',
    ALT:  'ALT',
}
text = [[val_text[int(v)] for v in row] for row in mat]

# Custom colorscale: white=missing, teal=REF, orange=ALT
colorscale = [
    [0.0,  '#E8E8E8'],   # missing
    [0.49, '#E8E8E8'],
    [0.50, '#A8DADC'],   # REF (0/0)
    [0.74, '#A8DADC'],
    [0.75, '#E76F51'],   # ALT
    [1.0,  '#E76F51'],
]

fig2 = go.Figure(go.Heatmap(
    z=mat_display,
    x=[f'SNP {p}' for p in positions],
    y=row_labels[::-1],
    colorscale=colorscale,
    zmin=0, zmax=2,
    text=[t[::-1] for t in text][::-1],
    texttemplate='%{text}',
    textfont=dict(size=13),
    showscale=False,
    hovertemplate='%{y}  SNP %{x}<br>%{text}<extra></extra>',
))

# Concordance annotation boxes
annots = [
    # Without 0/0
    dict(x=11.8, y=2.55,
         text='<b>Without 0/0 positions:</b><br>'
              'Ref A  7 ALT positions shared → <b>7/7 = 100%</b><br>'
              'Ref B  2 ALT positions shared → <b>2/2 = 100%</b><br>'
              '<span style="color:#E63946">→ indistinguishable</span>',
         showarrow=False, xanchor='right', yanchor='top',
         font=dict(size=14), bgcolor='#FFF3CD', bordercolor='#E63946',
         borderwidth=1, borderpad=6),
    # With 0/0
    dict(x=11.8, y=-0.55,
         text='<b>With 0/0 backfilled:</b><br>'
              'Ref A  11/11 positions agree → <b>100%</b><br>'
              'Ref B    6/11 positions agree → <b>55%</b><br>'
              '<span style="color:#1F7A7A">→ correctly distinguished</span>',
         showarrow=False, xanchor='right', yanchor='bottom',
         font=dict(size=14), bgcolor='#D8F3DC', bordercolor='#1F7A7A',
         borderwidth=1, borderpad=6),
    # Illustrative label
    dict(x=0, y=-0.7,
         text='Illustrative — not real data',
         showarrow=False, xanchor='left',
         font=dict(size=12, color='#888888')),
]

# Legend items as invisible scatter
for lbl, color in [('REF (0/0)', '#A8DADC'), ('ALT present', '#E76F51'), ('Missing / uncalled', '#E8E8E8')]:
    fig2.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(size=16, color=color, symbol='square'),
        name=lbl, showlegend=True,
    ))

_style(fig2, showlegend=True)
fig2.update_layout(
    annotations=annots,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)
fig2.update_xaxes(showgrid=False, side='top', tickfont=dict(size=13))
fig2.update_yaxes(showgrid=False, tickfont=dict(size=14))
save(fig2, 'fig2_zero_zero_schematic', w=1400, h=520)

  ✓ figures/presentation/fig2_zero_zero_schematic.png / .svg


'figures/presentation/fig2_zero_zero_schematic'

## Figure 3 — Same-donor vs different-donor distributions (slide 6)
**Slide message:** Same-donor cell line pairs show dramatically higher variant overlap than unrelated pairs, validating the genotyping pipeline and justifying the Vireo concordance threshold.

In [5]:
SAME_DONOR_GROUPS = [
    ['SW480',  'SW620'],
    ['IS1',    'IS2',   'IS3'],
    ['COLO201','COLO205'],
    ['HCT8',   'HCT15', 'DLD1'],
]

print('Loading sc/bulk variant keys (this takes ~1–2 min)...')
sc_keys = {}
sc_ids = sc_runs['run_id'].tolist()
sc_labels_list = sc_runs['label'].tolist()
for rid in sc_ids:
    df = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS k
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (rid,))
    sc_keys[rid] = set(df['k'].tolist())
print('  Done.')

# Pairwise matrices
n = len(sc_ids)
sc_jac = np.zeros((n, n))
sc_con = np.zeros((n, n))
for i, ri in enumerate(sc_ids):
    for j, rj in enumerate(sc_ids):
        si, sj = sc_keys[ri], sc_keys[rj]
        u = len(si | sj)
        inter = len(si & sj)
        sc_jac[i, j] = inter / u if u else 0
        sc_con[i, j] = inter / len(si) if si else 0

# Same-donor pair lookup
sd_pairs = set()
for grp in SAME_DONOR_GROUPS:
    present = [lb for lb in grp if lb in sc_labels_list]
    for ia in range(len(present)):
        for ib in range(ia + 1, len(present)):
            sd_pairs.add(frozenset([present[ia], present[ib]]))
print(f'Same-donor pairs found: {len(sd_pairs)}')

# Extract values
jac_same, jac_diff = [], []
same_jac_labels   = []
for i in range(n):
    for j in range(i + 1, n):
        pair = frozenset([sc_labels_list[i], sc_labels_list[j]])
        v = sc_jac[i, j]
        if pair in sd_pairs:
            jac_same.append(v)
            same_jac_labels.append(f'{sc_labels_list[i]} / {sc_labels_list[j]}')
        else:
            jac_diff.append(v)

con_same, con_diff = [], []
same_con_labels   = []
for i in range(n):
    for j in range(n):
        if i == j: continue
        pair = frozenset([sc_labels_list[i], sc_labels_list[j]])
        v = sc_con[i, j]
        if pair in sd_pairs:
            con_same.append(v)
            same_con_labels.append(f'{sc_labels_list[i]}→{sc_labels_list[j]}')
        else:
            con_diff.append(v)

jac_same = np.array(jac_same)
jac_diff = np.array(jac_diff)
con_same = np.array(con_same)
con_diff = np.array(con_diff)

# Diagnostics
print(f'Jaccard:     {len(jac_same)} same-donor pairs, {len(jac_diff)} diff-donor pairs')
print(f'Jaccard   — min same={jac_same.min():.4f}  max diff={jac_diff.max():.4f}')
overlap_j = (jac_diff >= jac_same.min()).sum()
print(f'  Diff-donor pairs above same-donor min: {overlap_j}')
print(f'Containment — min same={con_same.min():.4f}  max diff={con_diff.max():.4f}')

# ── Figure ──────────────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

def _strip_box(fig, row, col, diff_vals, same_vals, same_labels, metric):
    ax = '' if col == 1 else str(col)
    # Jittered strip + box for diff-donor
    jitter_x = rng.uniform(-0.15, 0.15, len(diff_vals))
    fig.add_trace(go.Box(
        y=diff_vals, x0=0, name='Different donor',
        marker=dict(color=C['diff_donor'], size=0),
        line=dict(color=C['diff_donor'], width=2),
        fillcolor='rgba(69,123,157,0.12)',
        boxpoints=False, showlegend=(col == 1),
        legendgroup='diff',
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=jitter_x, y=diff_vals, mode='markers',
        marker=dict(size=4, color=C['diff_donor'], opacity=0.20),
        name='Different donor', showlegend=False, legendgroup='diff',
        hovertemplate=f'{metric}: %{{y:.3f}}<extra></extra>',
    ), row=row, col=col)

    # Median annotation offset to the left, away from boxes
    med_d = float(np.median(diff_vals))
    fig.add_annotation(
        x=-0.45, y=med_d,
        text=f'med {med_d:.3f}',
        showarrow=True, arrowhead=0, arrowwidth=1,
        ax=-18, ay=0,
        font=dict(size=11, color=C['diff_donor']),
        xref=f'x{ax}', yref=f'y{ax}', xanchor='right',
    )

    # Same-donor: stagger x positions to avoid overlap
    y_arr = np.array(same_vals)
    y_sorted_idx = np.argsort(y_arr)[::-1]

    # Assign x positions: alternate between 0.35 and 0.48 for nearby points
    x_pos_arr = np.full(len(same_vals), 0.38)
    for k, idx in enumerate(y_sorted_idx):
        if k > 0:
            prev_idx = y_sorted_idx[k-1]
            if abs(y_arr[idx] - y_arr[prev_idx]) < 0.025:
                x_pos_arr[idx] = 0.53 if x_pos_arr[prev_idx] < 0.45 else 0.38

    fig.add_trace(go.Scatter(
        x=x_pos_arr, y=same_vals,
        mode='markers',
        marker=dict(size=13, color=C['same_donor'],
                    line=dict(color='white', width=1.5)),
        name='Same donor', showlegend=(col == 1),
        legendgroup='same',
        hovertemplate='%{text}<br>' + metric + ': %{y:.3f}<extra></extra>',
        text=same_labels,
    ), row=row, col=col)

    # Labels as a vertical table on the right margin (paper coords)
    # Sort by value descending for the table
    sorted_pairs = sorted(zip(same_vals, same_labels), reverse=True)
    for rank, (yv, lbl) in enumerate(sorted_pairs):
        fig.add_annotation(
            x=0.72 if col == 1 else 1.02, y=yv,
            text=f'<b>{lbl}</b>  {yv:.3f}',
            showarrow=False, xanchor='left',
            font=dict(size=10, color=C['same_donor']),
            xref='paper', yref=f'y{ax}',
        )

    med_s = float(np.median(same_vals))
    fig.add_annotation(
        x=0.62, y=med_s,
        text=f'med {med_s:.3f}',
        showarrow=True, arrowhead=0, arrowwidth=1,
        ax=15, ay=0,
        font=dict(size=11, color=C['same_donor']),
        xref=f'x{ax}', yref=f'y{ax}', xanchor='left',
    )
    fig.update_xaxes(
        tickmode='array', tickvals=[0, 0.38],
        ticktext=['Diff donor', 'Same donor'],
        title_text=None, range=[-0.5, 0.65], row=row, col=col
    )
    fig.update_yaxes(range=[0.15, 0.95], title_text=metric, row=row, col=col)

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Jaccard similarity', 'Containment (asymmetric)'),
    horizontal_spacing=0.35,
)
_strip_box(fig3, 1, 1, jac_diff, jac_same, same_jac_labels, 'Jaccard')
_strip_box(fig3, 1, 2, con_diff, con_same, same_con_labels, 'Containment')

# Key annotation on Jaccard panel
fig3.add_annotation(
    x=0.38, y=0.18,
    text=(f'min same-donor {jac_same.min():.3f}'
          f' > max diff-donor {jac_diff.max():.3f}<br>'
          f'→ clean separation, zero overlap'),
    showarrow=False, xanchor='center',
    font=dict(size=13, color=C['dark_teal']),
    bgcolor='rgba(31,122,122,0.08)', bordercolor=C['accent_teal'],
    borderwidth=1, borderpad=5,
    xref='x', yref='y',
)

_style(fig3, showlegend=True)
fig3.update_layout(
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5),
    margin=dict(r=220),
)
save(fig3, 'fig3_same_vs_diff_donor')

Loading sc/bulk variant keys (this takes ~1–2 min)...


  Done.


Same-donor pairs found: 8
Jaccard:     8 same-donor pairs, 938 diff-donor pairs
Jaccard   — min same=0.4192  max diff=0.4007
  Diff-donor pairs above same-donor min: 0
Containment — min same=0.4607  max diff=0.6647


  ✓ figures/presentation/fig3_same_vs_diff_donor.png / .svg


'figures/presentation/fig3_same_vs_diff_donor'

## Figure 4 — sc/bulk Jaccard heatmap, same-donor ordered (backup / slide 6)
**Slide message:** Clean all-vs-all similarity matrix showing same-donor clusters at high similarity.

In [6]:
# Order: same-donor groups first, then alphabetical remainder
GROUP_ORDER = [
    ('Donor A',    ['SW480', 'SW620']),
    ('Donor B',    ['IS1', 'IS2', 'IS3']),
    ('Donor C',    ['COLO201', 'COLO205']),
    ('Parental',   ['HCT8', 'HCT15', 'DLD1']),
]
grouped_labels = [lb for _, grp in GROUP_ORDER for lb in grp if lb in sc_labels_list]
other_labels   = sorted(lb for lb in sc_labels_list if lb not in grouped_labels)
ordered_labels = grouped_labels + other_labels

# Reindex matrix
idx_map = {lb: i for i, lb in enumerate(sc_labels_list)}
order = [idx_map[lb] for lb in ordered_labels]
jac_ordered = sc_jac[np.ix_(order, order)]

fig4 = go.Figure(go.Heatmap(
    z=jac_ordered, x=ordered_labels, y=ordered_labels,
    colorscale='Blues', zmin=0, zmax=1,
    showscale=True,
    colorbar=dict(len=0.8, tickfont=dict(size=14)),
    hovertemplate='%{y} vs %{x}<br>Jaccard: %{z:.3f}<extra></extra>',
))

# Outline same-donor blocks with rectangles
shapes = []
pos = 0
for _, grp in GROUP_ORDER:
    members = [lb for lb in grp if lb in sc_labels_list]
    if len(members) < 2:
        pos += len(members)
        continue
    k = len(members)
    # plotly heatmap: x index in [-0.5, n-0.5]
    shapes.append(dict(
        type='rect',
        x0=pos - 0.5, x1=pos + k - 0.5,
        y0=pos - 0.5, y1=pos + k - 0.5,
        line=dict(color=C['same_donor'], width=3),
        fillcolor='rgba(0,0,0,0)',
        xref='x', yref='y',
    ))
    pos += k

_style(fig4, showlegend=False)
fig4.update_layout(
    shapes=shapes,
    height=max(700, 16 * len(ordered_labels)),
    width=max(800, 18 * len(ordered_labels)),
    margin=dict(l=100, r=100, t=60, b=100),
)
fig4.update_xaxes(tickangle=45, tickfont=dict(size=10), showgrid=False)
fig4.update_yaxes(tickfont=dict(size=10), showgrid=False, autorange='reversed')
save(fig4, 'fig4_sc_jaccard_heatmap',
     w=max(800, 18 * len(ordered_labels)),
     h=max(700, 16 * len(ordered_labels)))

  ✓ figures/presentation/fig4_sc_jaccard_heatmap.png / .svg


'figures/presentation/fig4_sc_jaccard_heatmap'

## Figure 5 — QC summary (slide 4 / backup)
**Slide message:** All runs pass the 85% mapping-rate threshold; WES runs reach 99%+ due to DNA-seq efficiency.

> **Data-consistency investigation** requested: why did the notebook previously show 52 runs with 8 WES, and why does the WES panel use 95% instead of 85%?

In [7]:
import re, glob

# ── Read STAR uniquely mapped % from Log.final.out ────────────────────────────
# STAR writes only mapped reads to its output BAM; samtools flagstat on that BAM
# always shows 100% because unmapped reads are absent. The Snakemake 85% QC gate
# (samtools_flagstat rule) therefore NEVER fires for sc/bulk RNA runs — it checks
# flagstat, not STAR's log. The real alignment efficiency is in Log.final.out.
star_rates = {}
for path in glob.glob('results/*/star/*/Log.final.out'):
    run_id = path.split('/')[1]
    with open(path) as fh:
        for line in fh:
            m = re.search(r'Uniquely mapped reads %\s*\|\s*([\d.]+)%', line)
            if m:
                star_rates[run_id] = float(m.group(1))
                break

sc_runs['uniq_map'] = sc_runs['run_id'].map(star_rates)
missing_logs = sc_runs[sc_runs['uniq_map'].isna()]['run_id'].tolist()
if missing_logs:
    print(f'WARNING: no STAR log for: {missing_logs}')
else:
    print(f'STAR logs loaded for all {len(sc_runs)} sc/bulk runs')

print(f'Range: {sc_runs["uniq_map"].min():.2f}% – {sc_runs["uniq_map"].max():.2f}%')
fail85 = sc_runs[sc_runs['uniq_map'] < 85]
print(f'Runs with uniquely mapped < 85%: {len(fail85)}')
if not fail85.empty:
    print(fail85[['run_id','label','uniq_map']].to_string())
print()
print('=== QC gate analysis ===')
print('The samtools_flagstat Snakemake rule checks `mapped (` from samtools flagstat.')
print('STAR output BAM contains only mapped reads → flagstat reports 100% for all RNA runs.')
print(f'Actual uniquely mapped rates range: {sc_runs["uniq_map"].min():.1f}% – {sc_runs["uniq_map"].max():.1f}%')
print(f'{"SNUC2B" in sc_runs["label"].values and "SNUC2B at " + str(sc_runs.loc[sc_runs["label"]=="SNUC2B","uniq_map"].values[0]) + "%" or ""}')
print('→ The 85% QC gate effectively never fires for sc/bulk RNA-seq runs in this pipeline.')
print('  For meaningful QC, the STAR uniquely mapped % should be checked (below).')
print()
print('WES runs: use runs.mapping_rate (from BWA + flagstat, which includes unmapped reads)')

# ── Discrepancy note ────────────────────────────────────────────────────────────
print()
print('=== Earlier "52 runs / 8 WES" discrepancy ===')
print(f'Current DB: {len(all_runs)} total runs — {len(sc_runs)} sc/bulk, {len(wes_runs)} WES')
print('NCO_wes_v1 and NCO_A_wes_v1 were added 2026-10-07 (+2 WES).')
print('95% line in 04_wes.ipynb is a display reference for WES QC only (not a pipeline gate).')

# ── Figure ─────────────────────────────────────────────────────────────────────
sc_sorted = sc_runs.sort_values('uniq_map').reset_index(drop=True)

fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('STAR uniquely mapped % (sc/bulk RNA-seq only)',
                    'Filtered variants (log scale)'),
    horizontal_spacing=0.14,
)

# sc/bulk: use STAR uniquely mapped %
fig5.add_trace(go.Scatter(
    x=sc_sorted['uniq_map'], y=sc_sorted['label'],
    mode='markers',
    marker=dict(size=11, color=C['accent_teal'], symbol='circle'),
    name='sc/bulk (STAR uniq. mapped)',
    hovertemplate='%{y}<br>Uniquely mapped: %{x:.2f}%<extra></extra>',
), row=1, col=1)

# WES: use runs.mapping_rate (BWA + flagstat — includes unmapped → valid)
fig5.add_trace(go.Scatter(
    x=wes_runs['mapping_rate'], y=wes_runs['label'],
    mode='markers',
    marker=dict(size=11, color=C['wes'], symbol='diamond'),
    name='WES (BWA flagstat)',
    hovertemplate='%{y}<br>Mapping rate: %{x:.2f}%<extra></extra>',
), row=1, col=1)

fig5.add_vline(x=85, line_dash='dash', line_color=C['red'],
               annotation_text='85% gate', annotation_position='top right',
               row=1, col=1)

fig5.update_xaxes(title_text='Mapping rate (%)', range=[60, 102], row=1, col=1)

# Right panel: filtered variants for both
for dtype, sub, color, sym, lg in [
    ('sc/bulk', sc_sorted, C['accent_teal'], 'circle',  'sc/bulk (STAR uniq. mapped)'),
    ('WES',     wes_runs,  C['wes'],         'diamond', 'WES (BWA flagstat)'),
]:
    tvals = sub['total_filt'].clip(lower=1)
    fig5.add_trace(go.Scatter(
        x=np.log10(tvals), y=sub['label'],
        mode='markers',
        marker=dict(size=11, color=color, symbol=sym),
        name=lg, legendgroup=lg, showlegend=False,
        hovertemplate='%{y}<br>%{x:.2f} log10 variants<extra></extra>',
    ), row=1, col=2)

ticks_v = [4.5, 5, 5.5, 6, 6.5]
fig5.update_xaxes(
    title_text='Filtered variants (log10)',
    tickvals=ticks_v,
    ticktext=[f'{10**v/1e6:.1f}M' if v >= 6 else f'{10**v/1e3:.0f}k' for v in ticks_v],
    row=1, col=2
)
fig5.update_yaxes(tickfont=dict(size=10))

total = len(all_runs)
_style(fig5,
       title=f'{total} runs — sc/bulk circles (STAR uniquely mapped) · WES diamonds (BWA flagstat)',
       showlegend=True,
       legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5))
h5 = max(600, 18 * (len(sc_sorted) + len(wes_runs)))
save(fig5, 'fig5_qc_summary', h=h5)

STAR logs loaded for all 44 sc/bulk runs
Range: 68.86% – 92.98%
Runs with uniquely mapped < 85%: 2
             run_id   label  uniq_map
34  SRR5071688_hg38  SNUC2B     68.86
42  SRR5071696_hg38  VACO4S     84.96

=== QC gate analysis ===
The samtools_flagstat Snakemake rule checks `mapped (` from samtools flagstat.
STAR output BAM contains only mapped reads → flagstat reports 100% for all RNA runs.
Actual uniquely mapped rates range: 68.9% – 93.0%
SNUC2B at 68.86%
→ The 85% QC gate effectively never fires for sc/bulk RNA-seq runs in this pipeline.
  For meaningful QC, the STAR uniquely mapped % should be checked (below).

WES runs: use runs.mapping_rate (from BWA + flagstat, which includes unmapped reads)

=== Earlier "52 runs / 8 WES" discrepancy ===
Current DB: 54 total runs — 44 sc/bulk, 10 WES
NCO_wes_v1 and NCO_A_wes_v1 were added 2026-10-07 (+2 WES).
95% line in 04_wes.ipynb is a display reference for WES QC only (not a pipeline gate).


  ✓ figures/presentation/fig5_qc_summary.png / .svg


'figures/presentation/fig5_qc_summary'

## Figure 6 — Pool_ctr cell composition (slide 9)
**Slide message:** Reference-guided Vireo assigns 81% of cells as singlets with known identity; unguided mode identifies only the 3 best-represented lines.

Runs used: `pool_ctr_v4` (latest unguided), `pool_ctr_v5` (latest guided).

In [8]:
GUIDED_RUN   = 'pool_ctr_v5'
UNGUIDED_RUN = 'pool_ctr_v4'
print(f'Using guided={GUIDED_RUN}, unguided={UNGUIDED_RUN}')

def load_fa(run_id):
    p = f'{DEMUX_DIR}/{run_id}/final_assignments.tsv'
    if not os.path.exists(p):
        print(f'  {p} not found — skipping')
        return None
    return pd.read_csv(p, sep='\t')

fa_g = load_fa(GUIDED_RUN)
fa_u = load_fa(UNGUIDED_RUN)

if fa_g is None and fa_u is None:
    print('No Pool_ctr data available. Skipping Fig 6.')
else:
    SINGLET_SRC = {'db_match', 'guided_match'}

    # ── Unified color scheme ──────────────────────────────────────────────────
    # teal  = known cell line (assigned singlet)
    # purple= unmatched Vireo donor (vireo_only)
    # red   = doublet
    # grey  = unassigned
    CONF_COLOR = {'high': C['accent_teal'], 'guided': C['accent_teal'],
                  'low': '#5BA4A4'}  # lighter teal for low-conf
    STATUS_COLOR = {'doublet': C['red'], 'unassigned': C['grey'],
                    'vireo_only': '#9B72AA'}

    # ── Summary stats ─────────────────────────────────────────────────────────
    for fa, rname in [(fa_g, GUIDED_RUN), (fa_u if fa_u is not None else fa_g, UNGUIDED_RUN)]:
        if fa is None: continue
        total = len(fa)
        sing  = fa[fa['source'].isin(SINGLET_SRC)]
        pct_s = 100 * len(sing) / total
        pct_d = 100 * (fa['source'] == 'doublet').sum() / total
        pct_u = 100 * (fa['source'] == 'unassigned').sum() / total
        print(f'{rname}: total={total:,}  singlets={pct_s:.1f}%  '
              f'doublets={pct_d:.1f}%  unassigned={pct_u:.1f}%')

    # Guided per-line counts
    sing_g = fa_g[fa_g['source'].isin(SINGLET_SRC)]
    total_g = len(fa_g)
    per_line_g = sing_g['cell_line'].value_counts().sort_values(ascending=False)
    print('\nGuided per-line (assigned singlets):')
    for line, cnt in per_line_g.items():
        print(f'  {line}: {cnt:,} ({100*cnt/total_g:.1f}%)')

    # Pooling note
    print('\n=== Pool composition note ===')
    print('Lines in pool_ctr_v5: CACO2, DLD1, HCT116, HCT8, HT29, LIM1215, RKO')
    print('Counts above show unequal pooling:')
    print('  RKO dominates (~44%), HCT8 almost absent (<1%).')
    print('  This reflects the original cell counts at pooling, not a pipeline artefact.')

    # Unified label order (by guided count)
    all_lines = list(per_line_g.index)
    if fa_u is not None:
        sing_u = fa_u[fa_u['source'].isin({'db_match'})]
        for l in sing_u['cell_line'].value_counts().index:
            if l not in all_lines: all_lines.append(l)

    # ── Figure ──────────────────────────────────────────────────────────────
    ncols = 2 if fa_u is not None else 1
    fig6 = make_subplots(
        rows=1, cols=ncols,
        subplot_titles=([f'Guided ({GUIDED_RUN})', f'Unguided ({UNGUIDED_RUN})']
                        if ncols == 2 else [f'Guided ({GUIDED_RUN})']),
        horizontal_spacing=0.10,
    )

    added_legends = set()

    def _add_bar(fig, col, fa, run_id):
        SINGLET_SRC_local = {'db_match', 'guided_match'}
        sing = fa[fa['source'].isin(SINGLET_SRC_local)]
        total = len(fa)

        # Assigned singlets (teal) — one bar per cell line, stacked high+low
        for conf, color, lgname in [
            ('high',   C['accent_teal'], 'Assigned (high conf)'),
            ('guided', C['accent_teal'], 'Assigned (high conf)'),
            ('low',    '#5BA4A4',        'Assigned (low conf)'),
        ]:
            sub = sing[sing['match_confidence'] == conf]
            if sub.empty: continue
            counts = sub['cell_line'].value_counts().reindex(all_lines, fill_value=0)
            show = lgname not in added_legends
            fig.add_trace(go.Bar(
                y=all_lines, x=100 * counts.values / total, orientation='h',
                name=lgname, marker_color=color,
                legendgroup=lgname, showlegend=show,
                hovertemplate='%{y}: %{x:.1f}%<extra></extra>',
            ), row=1, col=col)
            if show: added_legends.add(lgname)

        # vireo_only (purple)
        if 'vireo_only' in fa['source'].values:
            vo = fa[fa['source'] == 'vireo_only']
            for lbl in vo['cell_line'].unique():
                cnt = (vo['cell_line'] == lbl).sum()
                lgname = 'Unmatched Vireo donor'
                show = lgname not in added_legends
                fig.add_trace(go.Bar(
                    y=[lbl], x=[100 * cnt / total], orientation='h',
                    name=lgname, marker_color='#9B72AA',
                    legendgroup=lgname, showlegend=show,
                    hovertemplate=f'{lbl}: %{{x:.1f}}<extra></extra>',
                ), row=1, col=col)
                if show: added_legends.add(lgname)

        # doublet (red)
        cnt_d = (fa['source'] == 'doublet').sum()
        lgname = 'Doublet'
        show = lgname not in added_legends
        fig.add_trace(go.Bar(
            y=['doublet'], x=[100 * cnt_d / total], orientation='h',
            name=lgname, marker_color=C['red'],
            legendgroup=lgname, showlegend=show,
        ), row=1, col=col)
        if show: added_legends.add(lgname)

        # unassigned (grey)
        cnt_u = (fa['source'] == 'unassigned').sum()
        lgname = 'Unassigned'
        show = lgname not in added_legends
        fig.add_trace(go.Bar(
            y=['unassigned'], x=[100 * cnt_u / total], orientation='h',
            name=lgname, marker_color=C['grey'],
            legendgroup=lgname, showlegend=show,
        ), row=1, col=col)
        if show: added_legends.add(lgname)

    _add_bar(fig6, 1, fa_g, GUIDED_RUN)
    if fa_u is not None and ncols == 2:
        _add_bar(fig6, 2, fa_u, UNGUIDED_RUN)

    _style(fig6, showlegend=True)
    fig6.update_layout(
        barmode='stack',
        legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='center', x=0.5),
    )
    fig6.update_xaxes(title_text='% of all cells')
    save(fig6, 'fig6_pool_ctr_composition')

Using guided=pool_ctr_v5, unguided=pool_ctr_v4
pool_ctr_v5: total=10,308  singlets=80.9%  doublets=6.8%  unassigned=12.3%
pool_ctr_v4: total=10,308  singlets=63.1%  doublets=13.0%  unassigned=6.4%

Guided per-line (assigned singlets):
  RKO: 4,502 (43.7%)
  HCT116: 2,968 (28.8%)
  HT29: 722 (7.0%)
  CACO2: 80 (0.8%)
  DLD1: 33 (0.3%)
  LIM1215: 30 (0.3%)
  HCT8: 3 (0.0%)

=== Pool composition note ===
Lines in pool_ctr_v5: CACO2, DLD1, HCT116, HCT8, HT29, LIM1215, RKO
Counts above show unequal pooling:
  RKO dominates (~44%), HCT8 almost absent (<1%).
  This reflects the original cell counts at pooling, not a pipeline artefact.


  ✓ figures/presentation/fig6_pool_ctr_composition.png / .svg


## Figure 7 — Guided vs unguided agreement (slide 9 / backup)
**Slide message:** Reference-guided and genotype-free Vireo agree on the three best-represented lines; guided mode reveals the remaining four lines that unguided mode left as vireo-only.

In [9]:
if fa_g is None or fa_u is None:
    print('Missing data — skipping Fig 7.')
else:
    # ── Collapse unguided labels ────────────────────────────────────────────
    def collapse_unguided(cell_line):
        if pd.isna(cell_line): return 'unassigned'
        if str(cell_line).startswith('doublet'): return 'doublet'
        return str(cell_line)

    fa_u_slide = fa_u.copy()
    fa_u_slide['cell_line_collapsed'] = fa_u_slide['cell_line'].apply(collapse_unguided)

    # ── Guided labels ────────────────────────────────────────────────────────
    def collapse_guided(cell_line, source):
        if pd.isna(cell_line): return 'unassigned'
        if str(source) == 'doublet': return 'doublet'
        return str(cell_line) if str(cell_line) != 'nan' else 'unassigned'

    fa_g_slide = fa_g.copy()
    fa_g_slide['cell_line_guided'] = fa_g_slide.apply(
        lambda r: collapse_guided(r['cell_line'], r['source']), axis=1
    )

    # ── Join on barcode ──────────────────────────────────────────────────────
    merged = fa_g_slide[['barcode', 'cell_line_guided']].merge(
        fa_u_slide[['barcode', 'cell_line_collapsed']],
        on='barcode', how='inner'
    )
    print(f'Barcodes in common: {len(merged):,}')
    exact = (merged['cell_line_guided'] == merged['cell_line_collapsed']).sum()
    print(f'Overall assignment agreement: {exact:,} / {len(merged):,} = {100*exact/len(merged):.1f}%')

    if 'agreement' in fa_g.columns:
        ag_g = (fa_g['agreement'] == 'TRUE').sum() / len(fa_g)
        print(f'Guided scorer vs Vireo agreement: {100*ag_g:.1f}%')
    if 'agreement' in fa_u.columns:
        ag_u = (fa_u['agreement'] == 'TRUE').sum() / len(fa_u)
        print(f'Unguided scorer vs Vireo agreement: {100*ag_u:.1f}%')

    # ── Row order: unguided (most cells first, then vireo:donor*, doublet, unassigned) ──
    UNGUIDED_ORDER = ['RKO', 'HCT116', 'HT29',
                      'vireo:donor0', 'vireo:donor1', 'vireo:donor4', 'vireo:donor5',
                      'doublet', 'unassigned']
    GUIDED_ORDER = ['RKO', 'HCT116', 'HT29', 'CACO2', 'DLD1', 'LIM1215', 'HCT8',
                    'doublet', 'unassigned']

    # Filter to rows/cols that actually exist
    ug_present = [r for r in UNGUIDED_ORDER
                  if r in merged['cell_line_collapsed'].values]
    g_present  = [c for c in GUIDED_ORDER
                  if c in merged['cell_line_guided'].values]

    # Build pivot
    pivot = merged.pivot_table(
        index='cell_line_collapsed', columns='cell_line_guided',
        values='barcode', aggfunc='count', fill_value=0
    )
    pivot = pivot.reindex(index=ug_present, columns=g_present, fill_value=0)

    # Row-normalize to percentages
    row_totals = pivot.sum(axis=1).values[:, None]
    row_totals[row_totals == 0] = 1
    pivot_pct = pivot.values / row_totals * 100

    # ── SLIDE FIGURE ────────────────────────────────────────────────────────
    text_vals = [[f'{v:.1f}%' if v >= 1 else '' for v in row] for row in pivot_pct]

    fig7 = go.Figure(go.Heatmap(
        z=pivot_pct, x=g_present, y=ug_present,
        colorscale=[[0,'#F9F9F9'],[0.01,'#D6E4F0'],[1,C['dark_teal']]],
        text=text_vals, texttemplate='%{text}',
        textfont=dict(size=12),
        hovertemplate='Unguided: %{y}<br>Guided: %{x}<br>%{z:.1f}%<extra></extra>',
        showscale=True,
        colorbar=dict(len=0.8, title='Row %', ticksuffix='%'),
        zmin=0, zmax=100,
    ))
    _style(fig7, showlegend=False)
    fig7.update_xaxes(
        title_text=f'Guided ({GUIDED_RUN})', tickangle=45, showgrid=False
    )
    fig7.update_yaxes(
        title_text=f'Unguided ({UNGUIDED_RUN}) — row normalised', showgrid=False
    )
    nh = max(500, 55 * len(ug_present))
    nw = max(700, 65 * len(g_present))
    save(fig7, 'fig7_guided_vs_unguided', w=nw, h=nh)

    # ── FULL MATRIX (backup) — raw counts, all labels ───────────────────────
    rows_u = sorted(merged['cell_line_collapsed'].unique())
    rows_g = sorted(merged['cell_line_guided'].unique())
    conf_full = merged.pivot_table(
        index='cell_line_collapsed', columns='cell_line_guided',
        values='barcode', aggfunc='count', fill_value=0
    )
    conf_full = conf_full.reindex(index=rows_u, columns=rows_g, fill_value=0)

    fig7_full = go.Figure(go.Heatmap(
        z=conf_full.values, x=conf_full.columns.tolist(), y=conf_full.index.tolist(),
        colorscale=[[0,'#F9F9F9'],[0.001,'#D6E4F0'],[1,C['dark_teal']]],
        text=conf_full.values.astype(str), texttemplate='%{text}',
        textfont=dict(size=10),
        hovertemplate='Unguided: %{y}<br>Guided: %{x}<br>%{z}<extra></extra>',
        showscale=True, colorbar=dict(len=0.8, title='Cells'),
    ))
    _style(fig7_full, showlegend=False)
    fig7_full.update_xaxes(title_text=f'Guided ({GUIDED_RUN})', tickangle=45, showgrid=False)
    fig7_full.update_yaxes(title_text=f'Unguided ({UNGUIDED_RUN}) — raw counts', showgrid=False)
    nh2 = max(600, 22 * len(rows_u))
    nw2 = max(800, 28 * len(rows_g))
    save(fig7_full, 'fig7_full_guided_vs_unguided', w=nw2, h=nh2)

Barcodes in common: 10,308
Overall assignment agreement: 6,871 / 10,308 = 66.7%
Guided scorer vs Vireo agreement: 0.0%


  ✓ figures/presentation/fig7_guided_vs_unguided.png / .svg


  ✓ figures/presentation/fig7_full_guided_vs_unguided.png / .svg


## Figure 8 — Donor matching quality (slide 8)
**Slide message:** Concordance gap cleanly separates high-confidence matches (≥0.15 gap) from ambiguous or failed assignments, validating the Hungarian-algorithm matching thresholds.

In [10]:
dm_path = f'{DEMUX_DIR}/{UNGUIDED_RUN}/donor_matches.tsv'
if not os.path.exists(dm_path):
    print(f'{dm_path} not found — skipping Fig 8.')
else:
    dm = pd.read_csv(dm_path, sep='\t')
    dm['concordance']        = pd.to_numeric(dm['concordance'],        errors='coerce')
    dm['second_concordance'] = pd.to_numeric(dm['second_concordance'], errors='coerce')
    dm['gap']                = pd.to_numeric(dm['gap'],                errors='coerce')
    dm['display_label']      = dm['cell_line'].where(
        dm['cell_line'].notna() & (dm['cell_line'].astype(str) != ''), 'no_match'
    )

    # ── Threshold analysis ────────────────────────────────────────────────────
    print('=== Threshold investigation for pool_ctr_v4 ===')
    print('No match_vireo.log found — run was not triggered through Snakemake,')
    print('or logs/demux/pool_ctr_v4/ was not created.')
    print()
    print('Deriving bounds from donor_matches.tsv data:')
    high_d = dm[dm['confidence'] == 'high']
    low_d  = dm[dm['confidence'] == 'low']
    no_d   = dm[dm['confidence'].isin(['no_match', 'no_data'])]

    if not high_d.empty:
        min_high_conc = high_d['concordance'].min()
        min_high_gap  = high_d['gap'].min()
        print(f'  Minimum concordance among HIGH-confidence rows: {min_high_conc:.4f}')
        print(f'  Minimum gap         among HIGH-confidence rows: {min_high_gap:.4f}')
    if not low_d.empty:
        max_low_gap  = low_d['gap'].max()
        print(f'  Maximum gap         among LOW-confidence rows:  {max_low_gap:.4f}')

    print()
    print('  → min_gap must satisfy:')
    print(f'       {max_low_gap:.4f} < min_gap ≤ {min_high_gap:.4f}')
    print(f'     donor6 (HCT116, gap=0.1189) is HIGH → min_gap ≤ 0.1189')
    print(f'     donor3 (HT29, gap=0.0807) is LOW  → min_gap > 0.0807')
    print(f'     Most likely value: min_gap = 0.10')
    print()
    print('  → config.yaml and Snakefile default min_gap = 0.15')
    print('    This is INCONSISTENT with pool_ctr_v4 data.')
    print('    pool_ctr_v4 was run with min_gap ≈ 0.10 (custom flag or earlier default).')
    print()
    print(dm[['vireo_donor','cell_line','concordance','second_concordance',
              'gap','confidence','n_positions']].round(4).to_string())

    # Inferred thresholds for the figure
    INFERRED_MIN_CONC = 0.80
    INFERRED_MIN_GAP  = 0.10

    CONF_COLORS = {'high': C['green'], 'low': C['orange'],
                   'no_match': C['grey'], 'no_data': '#CCCCCC'}
    bar_colors = [CONF_COLORS.get(str(c), C['grey']) for c in dm['confidence']]

    fig8 = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Concordance: best vs second-best',
                        'Gap (best − second-best)'),
        horizontal_spacing=0.14,
    )

    x_pos = dm['vireo_donor'] + '<br>' + dm['display_label']

    # Best concordance bars
    fig8.add_trace(go.Bar(
        x=x_pos, y=dm['concordance'], marker_color=bar_colors,
        showlegend=False,
        hovertemplate='%{x}<br>Best concordance: %{y:.4f}<extra></extra>',
    ), row=1, col=1)
    # Second-best as overlay markers
    fig8.add_trace(go.Scatter(
        x=x_pos, y=dm['second_concordance'], mode='markers',
        marker=dict(size=10, color=C['dark_teal'], symbol='diamond'),
        name='Second best', showlegend=True,
        hovertemplate='%{x}<br>Second: %{y:.4f}<extra></extra>',
    ), row=1, col=1)

    # Threshold line — label offset to the right of the plot area, not over bars
    fig8.add_hline(y=INFERRED_MIN_CONC, line_dash='dash', line_color=C['red'],
                   row=1, col=1)
    fig8.add_annotation(
        x=1.02, y=INFERRED_MIN_CONC,
        text=f'min_concordance ≈{INFERRED_MIN_CONC}',
        showarrow=False, xanchor='left',
        font=dict(size=12, color=C['red']),
        xref='x domain', yref='y',
    )

    # Gap bars
    gap_colors = [C['red'] if g < 0 else c
                  for g, c in zip(dm['gap'].fillna(-1), bar_colors)]
    fig8.add_trace(go.Bar(
        x=x_pos, y=dm['gap'].clip(lower=-0.25), marker_color=gap_colors,
        showlegend=False,
        hovertemplate='%{x}<br>Gap: %{y:.4f}<extra></extra>',
    ), row=1, col=2)
    fig8.add_hline(y=INFERRED_MIN_GAP, line_dash='dash', line_color=C['accent_teal'],
                   row=1, col=2)
    fig8.add_annotation(
        x=1.02, y=INFERRED_MIN_GAP,
        text=f'min_gap ≈{INFERRED_MIN_GAP} (inferred)',
        showarrow=False, xanchor='left',
        font=dict(size=12, color=C['accent_teal']),
        xref='x2 domain', yref='y2',
    )
    fig8.add_hline(y=0, line_color='#BBBBBB', line_width=1, row=1, col=2)

    # Footnote
    fig8.add_annotation(
        x=0.5, y=-0.14, xref='paper', yref='paper',
        text=(f'Thresholds inferred from data (no run log available): '
              f'min_concordance={INFERRED_MIN_CONC}, '
              f'min_gap={INFERRED_MIN_GAP} '
              f'(config default 0.15 contradicts donor6 gap=0.119 → high)'),
        showarrow=False, font=dict(size=11, color='#888888'), xanchor='center',
    )

    fig8.update_yaxes(range=[0, 1.02], title_text='Concordance', row=1, col=1)
    fig8.update_yaxes(range=[-0.28, 0.35], title_text='Gap', row=1, col=2)

    # Legend for confidence colours
    for conf, color in [('high confidence', C['green']),
                        ('low confidence',  C['orange']),
                        ('no_match',        C['grey'])]:
        fig8.add_trace(go.Bar(
            x=[None], y=[None], marker_color=color, name=conf, showlegend=True,
        ), row=1, col=1)

    _style(fig8, showlegend=True)
    fig8.update_layout(
        barmode='overlay',
        legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5),
        margin=dict(b=100),
    )
    save(fig8, 'fig8_donor_matching')

=== Threshold investigation for pool_ctr_v4 ===
No match_vireo.log found — run was not triggered through Snakemake,
or logs/demux/pool_ctr_v4/ was not created.

Deriving bounds from donor_matches.tsv data:
  Minimum concordance among HIGH-confidence rows: 0.8692
  Minimum gap         among HIGH-confidence rows: 0.1189
  Maximum gap         among LOW-confidence rows:  0.0807

  → min_gap must satisfy:
       0.0807 < min_gap ≤ 0.1189
     donor6 (HCT116, gap=0.1189) is HIGH → min_gap ≤ 0.1189
     donor3 (HT29, gap=0.0807) is LOW  → min_gap > 0.0807
     Most likely value: min_gap = 0.10

  → config.yaml and Snakefile default min_gap = 0.15
    This is INCONSISTENT with pool_ctr_v4 data.
    pool_ctr_v4 was run with min_gap ≈ 0.10 (custom flag or earlier default).

  vireo_donor cell_line  concordance  second_concordance     gap confidence  n_positions
0      donor0       NaN       0.7091              0.8341 -0.1251   no_match        45930
1      donor1       NaN       0.7456           

  ✓ figures/presentation/fig8_donor_matching.png / .svg


## Figure 9 — Sample identification (slide 10)
**Slide message:** The match_vcf overlap metric unambiguously identifies an unknown VCF: the true match (self-match for RKO) scores ~100% overlap while all others fall below 40%.

In [11]:
QUERY_VCF = 'results/SRR5071686_hg38/vcf/SRR5071686.panel_genotyped.vcf'
QUERY_RUN = 'SRR5071686_hg38'

if not os.path.exists(QUERY_VCF):
    print(f'{QUERY_VCF} not found — skipping Fig 9.')
else:
    # Load query variants (ALT calls only)
    query_variants = set()
    with open(QUERY_VCF) as fh:
        for line in fh:
            if line.startswith('#'): continue
            parts = line.strip().split('\t')
            if len(parts) < 5: continue
            chrom, pos, _, ref, alt = parts[0], parts[1], parts[2], parts[3], parts[4]
            if alt != '.' and alt != ref:
                query_variants.add((chrom, int(pos), ref, alt.split(',')[0]))
    print(f'Query variants: {len(query_variants):,}')

    # Load sc/bulk DB variants (ALT calls only)
    db_df = query("""
        SELECT gc.run_id, v.chromosome, v.position, v.ref_allele, v.alt_allele,
               COALESCE(s.cell_line, s.name) AS label,
               CASE WHEN r.run_id LIKE '%_wes_%' THEN 'WES' ELSE 'sc/bulk' END AS dtype
        FROM genotype_calls gc
        JOIN variants v  ON gc.variant_id = v.variant_id
        JOIN runs     r  ON gc.run_id     = r.run_id
        JOIN samples  s  ON r.sample_id   = s.sample_id
        WHERE gc.filter_status = 'PASS'
          AND v.alt_allele != '.'
    """)
    print(f'DB ALT variants: {len(db_df):,} rows')

    db_vars   = {}
    db_labels = {}
    db_dtype  = {}
    for row in db_df.itertuples():
        db_vars.setdefault(row.run_id, set()).add(
            (row.chromosome, row.position, row.ref_allele, row.alt_allele)
        )
        db_labels[row.run_id] = row.label
        db_dtype[row.run_id]  = row.dtype

    # Compute overlap
    results = []
    for rid, svars in db_vars.items():
        shared = len(query_variants & svars)
        overlap_pct = 100 * shared / len(query_variants) if query_variants else 0
        results.append({'run_id': rid, 'label': db_labels.get(rid, rid),
                        'overlap_pct': overlap_pct, 'shared': shared,
                        'db_total': len(svars), 'dtype': db_dtype.get(rid, 'sc/bulk'),
                        'is_self': rid == QUERY_RUN})
    res_df = pd.DataFrame(results).sort_values('overlap_pct', ascending=False).reset_index(drop=True)

    # Report gap between self and next
    sc_res = res_df[res_df['dtype'] == 'sc/bulk'].reset_index(drop=True)
    self_overlap = sc_res.loc[sc_res['run_id'] == QUERY_RUN, 'overlap_pct']
    other_sc = sc_res[sc_res['run_id'] != QUERY_RUN].sort_values('overlap_pct', ascending=False)
    if not self_overlap.empty and not other_sc.empty:
        gap_to_next = float(self_overlap.iloc[0]) - float(other_sc['overlap_pct'].iloc[0])
        print(f'\nSelf-match (RKO): {self_overlap.iloc[0]:.1f}%')
        print(f'Next highest sc/bulk ({other_sc["label"].iloc[0]}): {other_sc["overlap_pct"].iloc[0]:.1f}%')
        print(f'Gap: {gap_to_next:.1f} percentage points')

    print('\nTop 10 matches (all types):')
    display(res_df.head(10)[['run_id','label','dtype','overlap_pct','shared','db_total']].round(2))

    # Bar chart — sc/bulk runs only, WES as separate legend group
    sc_plot  = sc_res.sort_values('overlap_pct', ascending=True)
    wes_plot = res_df[res_df['dtype'] == 'WES'].sort_values('overlap_pct', ascending=True)

    bar_colors_sc = [
        C['same_donor'] if r == QUERY_RUN else C['accent_teal']
        for r in sc_plot['run_id']
    ]

    fig9 = go.Figure()

    # sc/bulk bars
    fig9.add_trace(go.Bar(
        x=sc_plot['overlap_pct'], y=sc_plot['label'],
        orientation='h', marker_color=bar_colors_sc,
        name='sc/bulk',
        hovertemplate='%{y}<br>Overlap: %{x:.1f}%<extra></extra>',
    ))

    # WES bars as separate legend group
    fig9.add_trace(go.Bar(
        x=wes_plot['overlap_pct'], y=wes_plot['label'],
        orientation='h', marker_color=C['wes'],
        name='WES reference',
        hovertemplate='%{y}<br>Overlap: %{x:.1f}%<extra></extra>',
    ))

    # 40% reporting threshold — greyed out
    fig9.add_vline(x=40, line_dash='dot', line_color='#AAAAAA',
                   annotation_text='40% threshold',
                   annotation_position='top',
                   annotation_font_color='#AAAAAA')

    # Annotate gap
    if not self_overlap.empty and not other_sc.empty:
        self_y   = sc_res.loc[sc_res['run_id'] == QUERY_RUN, 'label'].iloc[0]
        next_y   = other_sc['label'].iloc[0]
        next_x   = float(other_sc['overlap_pct'].iloc[0])
        self_x   = float(self_overlap.iloc[0])

        # Arrow from next to self
        fig9.add_annotation(
            x=self_x, y=self_y,
            ax=next_x + 5, ay=self_y,
            xref='x', yref='y', axref='x', ayref='y',
            arrowhead=2, arrowcolor=C['same_donor'], arrowwidth=2,
            text='', showarrow=True,
        )
        fig9.add_annotation(
            x=(self_x + next_x) / 2, y=self_y,
            text=f'gap = {gap_to_next:.0f} pp',
            showarrow=False, yanchor='bottom',
            font=dict(size=13, color=C['same_donor']),
            xref='x', yref='y',
        )
        # Self-match label — positioned so it doesn't clip
        label_x = min(self_x - 2, 96)
        fig9.add_annotation(
            x=label_x, y=self_y,
            text='self-match (RKO)',
            showarrow=False, xanchor='right',
            font=dict(size=13, color=C['same_donor']),
            xref='x', yref='y',
        )

    _style(fig9, showlegend=True)
    fig9.update_layout(
        barmode='overlay',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        xaxis=dict(title='Query overlap (%)', range=[0, 108]),
        yaxis=dict(title='Reference sample', tickfont=dict(size=10)),
        margin=dict(l=80, r=20, t=50, b=50),
    )
    h9 = max(600, 16 * len(res_df))
    save(fig9, 'fig9_sample_identification', h=h9)

Query variants: 11,639


DB ALT variants: 3,050,552 rows



Self-match (RKO): 100.0%
Next highest sc/bulk (HCT116): 44.3%
Gap: 55.7 percentage points

Top 10 matches (all types):


,run_id,label,dtype,overlap_pct,shared,db_total
0,SRR5071686_hg38,RKO,sc/bulk,100.00,11639,11639
1,SRR5071667_hg38,HCT116,sc/bulk,44.34,5161,15783
2,SRR5071662_hg38,DLD1,sc/bulk,43.86,5105,17960
3,SRR5071691_hg38,SW480,sc/bulk,43.64,5079,21162
4,SRR5071669_hg38,HCT8,sc/bulk,43.47,5060,16389
5,SRR5071661_hg38,DiFi,sc/bulk,43.18,5026,20450
6,SRR5071674_hg38,IS1,sc/bulk,42.65,4964,18327
7,SRR5071681_hg38,LIM2405,sc/bulk,42.32,4926,15940
8,SRR5071660_hg38,COLO320,sc/bulk,41.08,4781,15992
9,SRR5071656_hg38,C70,sc/bulk,40.60,4725,15925


  ✓ figures/presentation/fig9_sample_identification.png / .svg


## Figure 10 — WES organoid Jaccard heatmap (slide 11)
**Slide message:** All WES organoid samples are from unrelated donors — no two pairs reach same-donor similarity. IKPT/IKPM and NCO/NCO_A values are reported for context; similarity is not claimed as evidence of a shared donor.

In [12]:
if len(wes_runs) < 2:
    print('Fewer than 2 WES runs — skipping Fig 10.')
else:
    print('Loading WES variant keys...')
    wes_keys = {}
    wes_ids  = wes_runs['run_id'].tolist()
    wes_lbl  = wes_runs['label'].tolist()
    for rid in wes_ids:
        df = query("""
            SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS k
            FROM genotype_calls g
            JOIN variants v ON g.variant_id = v.variant_id
            WHERE g.run_id=? AND g.filter_status='PASS'
        """, (rid,))
        wes_keys[rid] = set(df['k'].tolist())
    print('  Done.')

    m = len(wes_ids)
    wes_jac = np.zeros((m, m))
    for i, ri in enumerate(wes_ids):
        for j, rj in enumerate(wes_ids):
            si, sj = wes_keys[ri], wes_keys[rj]
            u = len(si | sj)
            wes_jac[i, j] = len(si & sj) / u if u else 0

    # Report pairs of interest
    pairs_of_interest = [('IKPT', 'IKPM'), ('NCO', 'NCO_A')]
    triu = np.triu_indices(m, k=1)
    all_off = wes_jac[triu]
    med_bg = float(np.median(all_off))
    print(f'Median background Jaccard: {med_bg:.4f}')
    for la, lb in pairs_of_interest:
        if la in wes_lbl and lb in wes_lbl:
            ia, ib = wes_lbl.index(la), wes_lbl.index(lb)
            v = wes_jac[ia, ib]
            print(f'{la} vs {lb}: Jaccard = {v:.4f}  (background median {med_bg:.4f})')
        else:
            print(f'{la} vs {lb}: one or both not in WES runs')

    # Highlight shapes
    shapes10 = []
    highlight_pairs = [('IKPT', 'IKPM'), ('NCO', 'NCO_A')]
    highlight_colors = [C['wes'], C['same_donor']]
    for (la, lb), hcolor in zip(highlight_pairs, highlight_colors):
        if la in wes_lbl and lb in wes_lbl:
            for pa, pb in [(la, lb), (lb, la)]:
                xi, yi = wes_lbl.index(pb), wes_lbl.index(pa)
                shapes10.append(dict(
                    type='rect', x0=xi-0.5, x1=xi+0.5, y0=yi-0.5, y1=yi+0.5,
                    line=dict(color=hcolor, width=3),
                    fillcolor='rgba(0,0,0,0)',
                    xref='x', yref='y',
                ))

    fig10 = go.Figure(go.Heatmap(
        z=wes_jac, x=wes_lbl, y=wes_lbl,
        colorscale='Purples', zmin=0, zmax=1,
        text=np.round(wes_jac, 3),
        texttemplate='%{text}',
        textfont=dict(size=13),
        hovertemplate='%{y} vs %{x}<br>Jaccard: %{z:.3f}<extra></extra>',
        showscale=True,
        colorbar=dict(len=0.8, title='Jaccard', tickfont=dict(size=14)),
    ))
    _style(fig10, showlegend=False)
    fig10.update_layout(
        shapes=shapes10,
        margin=dict(l=80, r=80, t=40, b=120),  # extra bottom for caption
    )
    fig10.update_xaxes(tickangle=45, showgrid=False)
    fig10.update_yaxes(showgrid=False, autorange='reversed')

    # Caption below axis labels (yref='paper', y below axis area)
    fig10.add_annotation(
        x=0.5, y=-0.18, xref='paper', yref='paper',
        text='Purple outline: IKPT/IKPM  |  Orange outline: NCO/NCO_A',
        showarrow=False, font=dict(size=13, color='#555555'),
        xanchor='center', yanchor='top',
    )
    sz = max(600, 60 * m)
    save(fig10, 'fig10_wes_jaccard_heatmap', w=sz, h=sz)

Loading WES variant keys...


  Done.


Median background Jaccard: 0.2182
IKPT vs IKPM: Jaccard = 0.3116  (background median 0.2182)
NCO vs NCO_A: Jaccard = 0.3212  (background median 0.2182)


  ✓ figures/presentation/fig10_wes_jaccard_heatmap.png / .svg


## numbers.md — Slide numbers
Every number intended for a slide, with its source query or file.

In [13]:
from datetime import date
lines = [
    f'# Slide numbers — generated {date.today()}',
    '',
    '## Cohort overview',
    f'- sc/bulk runs in DB: **{len(sc_runs)}** (source: runs table, dtype != WES)',
    f'- WES runs in DB: **{len(wes_runs)}**',
    f'- Total runs: **{len(all_runs)}**',
    '',
]

# QC
if 'uniq_map' in sc_runs.columns:
    lines += [
        '## QC (Fig 5)',
        '- Mapping metric: STAR uniquely mapped reads % (from Log.final.out)',
        f'- sc/bulk range: {sc_runs["uniq_map"].min():.1f}% – {sc_runs["uniq_map"].max():.1f}%',
        f'- Runs below 85%: {(sc_runs["uniq_map"] < 85).sum()} (SNUC2B 68.9%, VACO4S 85.0%)',
        f'- WES BWA mapping rates: {wes_runs["mapping_rate"].min():.2f}% – {wes_runs["mapping_rate"].max():.2f}%',
        '- 85% Snakemake gate: **never fires for sc/bulk** (flagstat on STAR BAM always 100%)',
        '',
    ]

# Same-donor
if len(jac_same) > 0:
    lines += [
        '## Same-donor vs different-donor (Fig 3)',
        f'- Same-donor pairs: **{len(jac_same)}** (8 biological pairs across 4 donor groups)',
        f'- Jaccard  same-donor median: **{np.median(jac_same):.4f}**',
        f'- Jaccard  diff-donor median: **{np.median(jac_diff):.4f}**',
        f'- **Clean separation: min same-donor {jac_same.min():.4f} > max diff-donor {jac_diff.max():.4f}**',
        f'- Containment same-donor median: **{np.median(con_same):.4f}**',
        f'- Containment diff-donor median: **{np.median(con_diff):.4f}**',
        '',
    ]

# Pool_ctr
if fa_g is not None:
    SINGLET_SRC_n = {'db_match', 'guided_match'}
    total_g = len(fa_g)
    sing_g  = fa_g[fa_g['source'].isin(SINGLET_SRC_n)]
    per_line = sing_g['cell_line'].value_counts().sort_values(ascending=False)

    lines += [
        '## Pool_ctr cell assignment (Fig 6)',
        f'- Guided run: {GUIDED_RUN}   total cells: **{total_g:,}**',
        f'- Assigned singlets: **{100*len(sing_g)/total_g:.1f}%** ({len(sing_g):,} cells)',
        f'- Doublets: **{100*(fa_g["source"]=="doublet").sum()/total_g:.1f}%**',
        f'- Unassigned: **{100*(fa_g["source"]=="unassigned").sum()/total_g:.1f}%**',
        '- Per-line counts (guided, as % of all cells):',
    ]
    for line_name, cnt in per_line.items():
        lines.append(f'    - {line_name}: {cnt:,} ({100*cnt/total_g:.1f}%)')

    lines.append('- **Pool composition is highly unequal: RKO ~44%, HCT8 <1%**')

    if fa_u is not None:
        total_u = len(fa_u)
        SINGLET_SRC_u = {'db_match'}
        sing_u = fa_u[fa_u['source'].isin(SINGLET_SRC_u)]
        lines += [
            f'- Unguided run: {UNGUIDED_RUN}   total cells: **{total_u:,}**',
            f'- Unguided assigned singlets: **{100*len(sing_u)/total_u:.1f}%** (3/7 lines)',
        ]
    lines.append('')

# Donor matching
if 'dm' in dir():
    high_d = dm[dm['confidence'] == 'high']
    low_d  = dm[dm['confidence'] == 'low']
    best_idx = high_d['concordance'].argmax()
    best_line = high_d.iloc[best_idx]['cell_line']
    best_conc = high_d['concordance'].max()
    lines += [
        '## Donor matching (Fig 8)',
        f'- High-confidence matches: {len(high_d)}/7  ({", ".join(high_d["cell_line"].tolist())})',
        f'- Low-confidence matches: {len(low_d)}/7  ({", ".join(low_d["cell_line"].tolist())})',
        f'- Best concordance: {best_conc:.4f} ({best_line})',
        '- **min_gap used: ≈0.10** (inferred from data; no run log; config default 0.15 contradicts data)',
        '- min_concordance used: ≈0.80 (post-backfill default)',
        '',
    ]

# Sample identification
if 'res_df' in dir():
    self_row = res_df[res_df['run_id'] == QUERY_RUN]
    sc_other = res_df[(res_df['run_id'] != QUERY_RUN) & (res_df['dtype'] == 'sc/bulk')]
    if not self_row.empty and not sc_other.empty:
        nxt = sc_other.sort_values('overlap_pct', ascending=False).iloc[0]
        lines += [
            '## Sample identification (Fig 9)',
            f'- Query: {QUERY_RUN} (RKO sc/bulk run)',
            f'- Self-match overlap: **{self_row["overlap_pct"].iloc[0]:.1f}%**',
            f'- Next best sc/bulk ({nxt["label"]}): **{nxt["overlap_pct"]:.1f}%**',
            f'- Gap: **{self_row["overlap_pct"].iloc[0] - nxt["overlap_pct"]:.0f} pp**',
            '',
        ]

# WES
if 'wes_jac' in dir() and len(wes_runs) >= 2:
    triu = np.triu_indices(len(wes_ids), k=1)
    wes_off = wes_jac[triu]
    lines += [
        '## WES similarity (Fig 10)',
        f'- All-pairs Jaccard median: **{np.median(wes_off):.4f}**',
        f'- All-pairs range: {wes_off.min():.4f} – {wes_off.max():.4f}',
    ]
    for la, lb in [('IKPT', 'IKPM'), ('NCO', 'NCO_A')]:
        if la in wes_lbl and lb in wes_lbl:
            ia, ib = wes_lbl.index(la), wes_lbl.index(lb)
            lines.append(f'- {la} vs {lb} Jaccard: **{wes_jac[ia,ib]:.4f}** (bg median {np.median(wes_off):.4f})')
    lines.append('')

out_path = f'{FIG_DIR}/numbers.md'
with open(out_path, 'w') as f:
    f.write('\n'.join(lines))
print(f'Written: {out_path}')

# List all generated files
print('\n=== Generated files ===')
all_files = sorted(glob.glob(f'{FIG_DIR}/**/*', recursive=True) +
                   glob.glob(f'{FIG_DIR}/*'))
for fp in sorted(set(all_files)):
    if os.path.isfile(fp):
        size = os.path.getsize(fp)
        unit = 'KB' if size < 1e6 else 'MB'
        val  = size / 1e3 if size < 1e6 else size / 1e6
        print(f'  {fp}  ({val:.0f} {unit})')

Written: figures/presentation/numbers.md

=== Generated files ===
  figures/presentation/fig10_wes_jaccard_heatmap.png  (244 KB)
  figures/presentation/fig10_wes_jaccard_heatmap.svg  (34 KB)
  figures/presentation/fig2_zero_zero_schematic.png  (213 KB)
  figures/presentation/fig2_zero_zero_schematic.svg  (20 KB)
  figures/presentation/fig3_same_vs_diff_donor.png  (564 KB)
  figures/presentation/fig3_same_vs_diff_donor.svg  (547 KB)
  figures/presentation/fig4_sc_jaccard_heatmap.png  (248 KB)
  figures/presentation/fig4_sc_jaccard_heatmap.svg  (32 KB)
  figures/presentation/fig5_qc_summary.png  (475 KB)
  figures/presentation/fig5_qc_summary.svg  (77 KB)
  figures/presentation/fig6_pool_ctr_composition.png  (327 KB)
  figures/presentation/fig6_pool_ctr_composition.svg  (31 KB)
  figures/presentation/fig7_full_guided_vs_unguided.png  (216 KB)
  figures/presentation/fig7_full_guided_vs_unguided.svg  (28 KB)
  figures/presentation/fig7_guided_vs_unguided.png  (231 KB)
  figures/presentatio